In [1]:
from google.colab import files

uploaded = files.upload()

Saving generation_monthly.xlsx to generation_monthly.xlsx


In [8]:
import pandas as pd
from io import BytesIO

filename = next(iter(uploaded))

excel_file = pd.ExcelFile(BytesIO(uploaded[filename]))

print("Worksheets:")
print(excel_file.sheet_names)

Worksheets:
['2001_2002_FINAL', '2003_2004_FINAL', '2005-2007_FINAL', '2008-2009_FINAL', '2010-2011_FINAL', '2012_Final', '2013_Final', '2014_Final', '2015_Final', '2016_Final', '2017_Final', '2018_Final', '2019_Final', '2020_Final', '2021_Final', '2022_Final', '2023_Final', '2024_Final', '2025_Final', '2026_Preliminary', 'EnergySource_Notes']


In [9]:
import pandas as pd
import re

excel_file = pd.ExcelFile(filename)

# Select worksheets whose names begin with a four-digit year
data_sheets = [
    sheet for sheet in excel_file.sheet_names
    if re.match(r"^\d{4}", sheet)
]

print("Data worksheets:")
for sheet in data_sheets:
    print(sheet)

print("\nExcluded worksheets:")
for sheet in excel_file.sheet_names:
    if sheet not in data_sheets:
        print(sheet)

Data worksheets:
2001_2002_FINAL
2003_2004_FINAL
2005-2007_FINAL
2008-2009_FINAL
2010-2011_FINAL
2012_Final
2013_Final
2014_Final
2015_Final
2016_Final
2017_Final
2018_Final
2019_Final
2020_Final
2021_Final
2022_Final
2023_Final
2024_Final
2025_Final
2026_Preliminary

Excluded worksheets:
EnergySource_Notes


In [11]:
import pandas as pd

all_data = []

for sheet in data_sheets:

    # Extract the first year from the sheet name
    year = int(sheet[:4])

    # 2012 and after the format of excel sheets change so have to treat differently
    if year < 2012:
        temp = pd.read_excel(excel_file, sheet_name=sheet, header=0)
    else:
        temp = pd.read_excel(excel_file, sheet_name=sheet, header=4)

    # Remove extra spaces from column names
    temp.columns = temp.columns.str.strip()

    # Record the original worksheet
    temp["source_sheet"] = sheet

    all_data.append(temp)

# Combine all worksheets
df = pd.concat(all_data, ignore_index=True)

print("Total rows:", len(df))
print("Columns:", df.columns.tolist())


# ensure displaying correctly
display(df.head())
display(df.tail())

Total rows: 605426
Columns: ['YEAR', 'MONTH', 'STATE', 'TYPE OF PRODUCER', 'ENERGY SOURCE', 'GENERATION (Megawatthours)', 'source_sheet', 'GENERATION\n(Megawatthours)']


,YEAR,MONTH,STATE,TYPE OF PRODUCER,ENERGY SOURCE,GENERATION (Megawatthours),source_sheet,GENERATION\n(Megawatthours)
0,2001,1,AK,Total Electric Power Industry,Coal,46903.0,2001_2002_FINAL,NaN
1,2001,1,AK,Total Electric Power Industry,Petroleum,71085.0,2001_2002_FINAL,NaN
2,2001,1,AK,Total Electric Power Industry,Natural Gas,367521.0,2001_2002_FINAL,NaN
3,2001,1,AK,Total Electric Power Industry,Hydroelectric Conventional,104549.0,2001_2002_FINAL,NaN
4,2001,1,AK,Total Electric Power Industry,Wind,87.0,2001_2002_FINAL,NaN


,YEAR,MONTH,STATE,TYPE OF PRODUCER,ENERGY SOURCE,GENERATION (Megawatthours),source_sheet,GENERATION\n(Megawatthours)
605421,2026,8,WY,"Electric Generators, Electric Utilities",Coal,NaN,2026_Preliminary,2321704.0
605422,2026,8,WY,"Electric Generators, Electric Utilities",Hydroelectric Conventional,NaN,2026_Preliminary,57639.0
605423,2026,8,WY,"Electric Generators, Electric Utilities",Natural Gas,NaN,2026_Preliminary,1002227.0
605424,2026,8,WY,"Electric Generators, Electric Utilities",Petroleum,NaN,2026_Preliminary,3318.0
605425,2026,8,WY,"Electric Generators, Electric Utilities",Wind,NaN,2026_Preliminary,331120.0


In [13]:
## ensuring consistent column format that is easier for sql to use

# Combine the two generation columns into one
df["generation_mwh"] = df[
    "GENERATION (Megawatthours)"
].combine_first(
    df["GENERATION\n(Megawatthours)"]
)

# Remove the original two columns
df = df.drop(columns=[
    "GENERATION (Megawatthours)",
    "GENERATION\n(Megawatthours)"
])

# Rename the remaining columns
df = df.rename(columns={
    "YEAR": "year",
    "MONTH": "month",
    "STATE": "state",
    "TYPE OF PRODUCER": "producer_type",
    "ENERGY SOURCE": "energy_source"
})

# Reorder columns
df = df[
    [
        "year",
        "month",
        "state",
        "producer_type",
        "energy_source",
        "generation_mwh",
        "source_sheet"
    ]
]

display(df.head())
print("\nMissing values:")
print(df.isna().sum())

,year,month,state,producer_type,energy_source,generation_mwh,source_sheet
0,2001,1,AK,Total Electric Power Industry,Coal,46903.0,2001_2002_FINAL
1,2001,1,AK,Total Electric Power Industry,Petroleum,71085.0,2001_2002_FINAL
2,2001,1,AK,Total Electric Power Industry,Natural Gas,367521.0,2001_2002_FINAL
3,2001,1,AK,Total Electric Power Industry,Hydroelectric Conventional,104549.0,2001_2002_FINAL
4,2001,1,AK,Total Electric Power Industry,Wind,87.0,2001_2002_FINAL



Missing values:
year              0
month             0
state             0
producer_type     0
energy_source     0
generation_mwh    0
source_sheet      0
dtype: int64


In [15]:
key_columns = [
    "year",
    "month",
    "state",
    "producer_type",
    "energy_source"
]

duplicates = df[df.duplicated(
    subset=key_columns,
    keep=False
)]

print("Duplicate key records:", len(duplicates))

display(duplicates.head(10))

Duplicate key records: 0


,year,month,state,producer_type,energy_source,generation_mwh,source_sheet


In [20]:
from google.colab import files

# Standardize text fields
for col in ["state", "producer_type", "energy_source"]:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# Keep the six columns needed for SQL
cleaned_df = df[
    [
        "year",
        "month",
        "state",
        "producer_type",
        "energy_source",
        "generation_mwh"
    ]
].copy()

# Check that standardizing text did not create duplicate keys
key_columns = [
    "year", "month", "state",
    "producer_type", "energy_source"
]

assert not cleaned_df.duplicated(subset=key_columns).any()

# Export
cleaned_df.to_csv("generation_cleaned.csv", index=False)

print("Rows exported:", len(cleaned_df))
files.download("generation_cleaned.csv")

Rows exported: 605426


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>